In [0]:
%sql
-- How has ADU activity trended over time?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_adu_growth_trend AS
WITH adu_permits AS (
    SELECT 
        dut.use_code,
        dut.use_desc,
        id.year AS permit_year,
        COUNT(*) AS total_permits,
        SUM(fp.du_changed) AS total_du_changed
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_use_type AS dut
        ON fp.use_type_key = dut.use_type_key
    LEFT JOIN la_lakehouse.gold.dim_date AS id
        ON fp.issue_date_key = id.date_key
    WHERE dut.use_desc IS NOT NULL 
      AND dut.use_desc LIKE '%Accessory Dwelling Unit%'
    GROUP BY 
        dut.use_code, 
        dut.use_desc, 
        id.year
)
SELECT 
    use_code,
    use_desc,
    permit_year,
    total_permits,
    total_du_changed,
    LAG(total_permits) OVER (
        PARTITION BY use_desc 
        ORDER BY permit_year
    ) AS prev_year_permit_count,
    ROUND(
        100.0 * (total_permits - LAG(total_permits, 1) OVER (PARTITION BY use_desc ORDER BY permit_year))
        / NULLIF(LAG(total_permits, 1) OVER (PARTITION BY use_desc ORDER BY permit_year), 0),
        2
    ) AS yoy_growth_pct
FROM adu_permits
ORDER BY 
    use_desc ASC, 
    permit_year ASC;


In [0]:
%sql
-- Which zones show the most net unit growth (du_changed)?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_zoning_density_summary AS
WITH categorized_permits AS (
    SELECT 
        fp.du_changed,
        CASE 
            WHEN dz.zone = 'C' OR dz.zone_base LIKE 'C%' THEN 'Commercial (Mixed-Use / High-Rise)'
            WHEN dz.zone = 'R' AND (dz.zone_base LIKE 'R4%' OR dz.zone_base LIKE 'R5%') THEN 'Multi-Family High-Density (R4/R5)'
            WHEN dz.zone = 'R' AND dz.zone_base LIKE 'R3%' THEN 'Multi-Family Medium-Density (R3)'
            WHEN dz.zone = 'RD' OR (dz.zone = 'R' AND dz.zone_base LIKE 'R2%') THEN 'Low-Density Duplex/Infill (R2/RD)'
            WHEN dz.zone IN ('R', 'RE', 'RA', 'RS') AND (dz.zone_base LIKE 'R1%' OR dz.zone_base LIKE 'RE%' OR dz.zone_base LIKE 'RA%' OR dz.zone_base LIKE 'RS%') THEN 'Single-Family / ADU (R1/RE/RA)'
            WHEN dz.zone IN ('M', 'MR', 'CM') THEN 'Commercial / Industrial Mixed-Use'
            ELSE 'Specific Plan / Special Overlays'
        END AS zoning_category
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_zone AS dz
        ON fp.zone_key = dz.zone_key
    WHERE dz.zone IS NOT NULL 
)
SELECT 
    zoning_category,
    SUM(du_changed) AS total_net_units,
    COUNT(*) AS total_housing_permits,
    ROUND(SUM(du_changed) * 1.0 / COUNT(*), 2) AS avg_net_units_per_permit,
    ROUND(100.0 * SUM(du_changed) / SUM(SUM(du_changed)) OVER (), 2) AS pct_share_of_total_growth
FROM categorized_permits
GROUP BY zoning_category
ORDER BY total_net_units DESC;

#Testing Tables

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_adu_growth;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vm_zoning_density_summary;